# Monitoring a served model

Once a model is [serving traffic](../08-persistence-deployment/serving-a-model.ipynb),
you have to watch it. Three **distinct** concerns, easily conflated:

1. **Operational health** — latency, error rate, request volume. Is the server
   up and responding?
2. **Data drift** — are incoming request features statistically different from
   the training distribution?
3. **Model/prediction drift** — is the distribution of predictions shifting, or
   (once ground truth arrives) is accuracy degrading?

Rust has no established MLOps-monitoring stack, so we build these from
general-purpose tools: `tracing`, `metrics`, and a hand-rolled drift check.

## Operational health: structured logging & metrics

Instrument each request with [`tracing`](https://docs.rs/tracing) (structured
logs) and [`metrics`](https://docs.rs/metrics) (counters/histograms). In the
[axum server](../08-persistence-deployment/serving-a-model.ipynb) this goes in
the handler; here we emit the same signals directly:

In [ ]:
:dep tracing = { version = "0.1" }
:dep tracing-subscriber = { version = "0.3" }
:dep metrics = { version = "0.24" }

tracing_subscriber::fmt::init();

// Per-request: a structured log line (request id, latency, prediction) ...
tracing::info!(request_id = 42, latency_ms = 8.1, prediction = 0.87, "handled /predict");
// ... and metrics a /metrics endpoint (via the prometheus exporter) would expose.
metrics::counter!("predict_requests_total").increment(1);
metrics::histogram!("predict_latency_ms").record(8.1);
println!("logged one request and recorded its metrics");

In production you'd install a Prometheus recorder (`metrics-exporter-prometheus`)
and expose a `GET /metrics` route alongside `/predict`; Prometheus scrapes it and
you alert on latency/error thresholds.

## Data drift: a Population Stability Index, by hand

**PSI** compares a live window of a feature against its training distribution,
bin by bin. It reuses the [EDA chapter's](../01b-eda/exploratory-data-analysis.ipynb)
distribution thinking, now on a rolling window of live requests. We deliberately
build it by hand — Rust has no mature drift-detection crate to lean on.

In [ ]:
// Proportion of values falling in each bin (bins defined by their edges).
fn proportions(values: &[f64], edges: &[f64]) -> Vec<f64> {
    let mut counts = vec![0.0_f64; edges.len() + 1];
    for &v in values {
        let mut b = 0;
        while b < edges.len() && v >= edges[b] { b += 1; }
        counts[b] += 1.0;
    }
    let total = values.len() as f64;
    counts.iter().map(|c| c / total).collect()
}

// PSI = sum over bins of (curr - ref) * ln(curr / ref).
fn psi(reference: &[f64], current: &[f64]) -> f64 {
    reference.iter().zip(current).map(|(&r, &c)| {
        let (r, c) = (r.max(1e-6), c.max(1e-6));
        (c - r) * (c / r).ln()
    }).sum()
}
println!("drift helpers ready");

In [ ]:
{
    let edges = [2.0, 4.0, 6.0];  // 4 bins: <2, 2-4, 4-6, >6
    // The feature's distribution when the model was trained.
    let training = vec![1.0,1.5,2.5,3.0,3.5,4.5,5.0,5.5,6.5,7.0];
    let reference = proportions(&training, &edges);

    // Two rolling windows of live requests.
    let live_stable = vec![1.2,2.8,3.2,4.8,5.2,6.8,1.8,3.8,5.8,2.2];
    let live_drift  = vec![6.5,7.0,6.8,7.5,6.2,7.8,6.9,7.1,6.6,7.3];

    let psi_stable = psi(&reference, &proportions(&live_stable, &edges));
    let psi_drift  = psi(&reference, &proportions(&live_drift, &edges));
    println!("reference proportions: {:?}", reference);
    println!("stable  window PSI = {:.3}  {}", psi_stable, flag(psi_stable));
    println!("drifted window PSI = {:.3}  {}", psi_drift, flag(psi_drift));
}

// Rule of thumb: < 0.1 no shift, 0.1-0.25 moderate, > 0.25 significant.
fn flag(psi: f64) -> &'static str {
    if psi < 0.1 { "(stable)" } else if psi < 0.25 { "(moderate drift)" } else { "(SIGNIFICANT DRIFT)" }
}

The stable window scores near zero; the shifted one trips the significant-drift
threshold — a signal to investigate.

## What to do when drift fires

Detection is the easy part; the response is a judgement call, and building it out
is beyond this book. The usual options: **alert** a human, **shift traffic**
gradually to a challenger model, or trigger **scheduled retraining**. A full
MLOps retraining pipeline is a natural next step, not something we implement here.

```{warning}
**Ecosystem maturity.** Unlike Python's MLOps tooling (Evidently, WhyLabs,
purpose-built drift libraries), Rust has no established, widely-adopted
equivalent. This chapter deliberately builds monitoring primitives **by hand** on
general-purpose crates (`tracing`, `metrics`, `axum`) rather than presenting an
ML-specific monitoring crate as mature, because none was found. Re-check the
ecosystem before relying on this.
```

Next: [Time Series](../10-time-series/time-series-fundamentals.ipynb) — a
different data shape, where ordering itself carries information.